<a href="https://colab.research.google.com/github/busycaesar/GPT/blob/Master/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Download the dataset to train on.
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

--2026-08-23 09:42:44--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.04s   

2026-08-23 09:42:44 (28.2 MB/s) - ‘input.txt’ saved [1115394/1115394]



In [2]:
print("Total characters:", len(text))

Total characters: 1115394


In [36]:
# All the unique characters that occur in the text
unique_characters = sorted(list(set(text)))
vocab_size = len(unique_characters)

In [38]:
print("".join(unique_characters))
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [5]:
# Mapping for each unique character.
stoi = { ch:i for i,ch in enumerate(unique_characters) }
itos = { i:ch for i,ch in enumerate(unique_characters) }

encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

In [6]:
print(encode("hii there"))
print(decode(encode("hii there")))

[46, 47, 47, 1, 58, 46, 43, 56, 43]
hii there


In [7]:
import torch

encoded_text = encode(text)

dataset = torch.tensor(encoded_text, dtype=torch.long)

In [8]:
print(dataset.shape, dataset.dtype)
print(dataset[:1000])

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59,  1, 39, 56, 43,  1, 39, 50, 50,
         1, 56, 43, 57, 53, 50, 60, 43, 42,  1, 56, 39, 58, 46, 43, 56,  1, 58,
        53,  1, 42, 47, 43,  1, 58, 46, 39, 52,  1, 58, 53,  1, 44, 39, 51, 47,
        57, 46, 12,  0,  0, 13, 50, 50, 10,  0, 30, 43, 57, 53, 50, 60, 43, 42,
         8,  1, 56, 43, 57, 53, 50, 60, 43, 42,  8,  0,  0, 18, 47, 56, 57, 58,
         1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 18, 47, 56, 57, 58,  6,  1, 63,
        53, 59,  1, 49, 52, 53, 61,  1, 15, 39, 47, 59, 57,  1, 25, 39, 56, 41,
      

In [9]:
# Split the data into train and validation sets

train_dataset_proportion = 0.9

number_of_dataset = int(train_dataset_proportion*len(dataset))

train_dataset = dataset[:number_of_dataset]
validation_dataset = dataset[number_of_dataset:]

In [10]:
block_size = 8

train_dataset[:block_size+1]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [19]:
x = train_dataset[:block_size]
y = train_dataset[1:block_size+1]

for tensor in range(block_size):
    context = x[:tensor+1]
    target = y[tensor]
    print(f"When input is {context} the expected target is {target}.")

When input is tensor([18]) the expected output is 47.
When input is tensor([18, 47]) the expected output is 56.
When input is tensor([18, 47, 56]) the expected output is 57.
When input is tensor([18, 47, 56, 57]) the expected output is 58.
When input is tensor([18, 47, 56, 57, 58]) the expected output is 1.
When input is tensor([18, 47, 56, 57, 58,  1]) the expected output is 15.
When input is tensor([18, 47, 56, 57, 58,  1, 15]) the expected output is 47.
When input is tensor([18, 47, 56, 57, 58,  1, 15, 47]) the expected output is 58.


In [30]:
# To maintain reproducability of random indices.
torch.manual_seed(1337)

# Parallel process on GPU
batch_size = 4
# Maximum context length for predicting next token
block_size = 8

def get_batch(split):
    dataset = train_dataset if split == 'train' else validation_dataset
    # Returns "batch_size" (4) random starting indices from the dataset.
    # The upper bound is "len(dataset) - block_size" (1003854 - 8) so that there are enough tokens for the block size even if the largest possible index is picked.
    ix = torch.randint(len(dataset) - block_size, (batch_size,))

    # Get "block_size" tokens starting at each chosen index, for all indices.
    context = torch.stack([dataset[i:i+block_size] for i in ix])

    # Get "block_size" tokens starting one position after each chosen index, for all indices.
    target = torch.stack([dataset[i+1:i+block_size+1] for i in ix])
    return context, target

In [32]:
context_batch, target_batch = get_batch('train')
print('inputs:')
print(context_batch)
print()
print('targets:')
print(target_batch)

print('----')

inputs:
tensor([[57, 43, 60, 43, 52,  1, 63, 43],
        [60, 43, 42,  8,  0, 25, 63,  1],
        [56, 42,  5, 57,  1, 57, 39, 49],
        [43, 57, 58, 63,  6,  1, 58, 46]])

targets:
tensor([[43, 60, 43, 52,  1, 63, 43, 39],
        [43, 42,  8,  0, 25, 63,  1, 45],
        [42,  5, 57,  1, 57, 39, 49, 43],
        [57, 58, 63,  6,  1, 58, 46, 47]])
----


In [33]:
for batch in range(batch_size):
    for tensor in range(block_size):
        context = context_batch[batch, :tensor+1]
        target = target_batch[batch,tensor]
        print(f"when input is {context.tolist()} the target: {target}")

when input is [57] the target: 43
when input is [57, 43] the target: 60
when input is [57, 43, 60] the target: 43
when input is [57, 43, 60, 43] the target: 52
when input is [57, 43, 60, 43, 52] the target: 1
when input is [57, 43, 60, 43, 52, 1] the target: 63
when input is [57, 43, 60, 43, 52, 1, 63] the target: 43
when input is [57, 43, 60, 43, 52, 1, 63, 43] the target: 39
when input is [60] the target: 43
when input is [60, 43] the target: 42
when input is [60, 43, 42] the target: 8
when input is [60, 43, 42, 8] the target: 0
when input is [60, 43, 42, 8, 0] the target: 25
when input is [60, 43, 42, 8, 0, 25] the target: 63
when input is [60, 43, 42, 8, 0, 25, 63] the target: 1
when input is [60, 43, 42, 8, 0, 25, 63, 1] the target: 45
when input is [56] the target: 42
when input is [56, 42] the target: 5
when input is [56, 42, 5] the target: 57
when input is [56, 42, 5, 57] the target: 1
when input is [56, 42, 5, 57, 1] the target: 57
when input is [56, 42, 5, 57, 1, 57] the targ

In [34]:
print(context_batch) # our input to the transformer

tensor([[57, 43, 60, 43, 52,  1, 63, 43],
        [60, 43, 42,  8,  0, 25, 63,  1],
        [56, 42,  5, 57,  1, 57, 39, 49],
        [43, 57, 58, 63,  6,  1, 58, 46]])


In [45]:
import torch
import torch.nn as nn
from torch.nn import functional as F

# To maintain reproducability of random weights.
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        # Lookup table mapping each token id to a row of vocab_size numbers. Each cell is assigned random values before training.
        # Normally, each token id is mapped to the row that contains the embedding (carrying semantic meaning) of the token.
        # The embeddings are then converted into logits to predict the next token.
        # Here, we skip all of that. The row length already equals vocab_size, so the row can be used directly as the logits.
        # So this model learns the logits directly in the table, instead of learning the many layers of weights that a real model uses to produce them.
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        # Gets the matrix of context batch and returns vocab_size logits for each item in the matrix.
        logits = self.token_embedding_table(idx) # (Batch Size, Block Size, Vocab Size)

        return logits

m = BigramLanguageModel(vocab_size)
print(context_batch)
out = m(context_batch)
print(out)
print(out.shape)

tensor([[57, 43, 60, 43, 52,  1, 63, 43],
        [60, 43, 42,  8,  0, 25, 63,  1],
        [56, 42,  5, 57,  1, 57, 39, 49],
        [43, 57, 58, 63,  6,  1, 58, 46]])
tensor([[[-0.5201,  0.2831,  1.0847,  ..., -0.0198,  0.7959,  1.6014],
         [ 0.3323, -0.0872, -0.7470,  ..., -0.6716, -0.9572, -0.9594],
         [-0.1679,  0.5602,  0.6467,  ...,  0.1522,  0.5109,  0.0990],
         ...,
         [ 0.5978, -0.0514, -0.0646,  ..., -1.4649, -2.0555,  1.8275],
         [-0.8109,  0.2410, -0.1139,  ...,  1.4509,  0.1836,  0.3064],
         [ 0.3323, -0.0872, -0.7470,  ..., -0.6716, -0.9572, -0.9594]],

        [[-0.1679,  0.5602,  0.6467,  ...,  0.1522,  0.5109,  0.0990],
         [ 0.3323, -0.0872, -0.7470,  ..., -0.6716, -0.9572, -0.9594],
         [ 1.0726,  0.7295, -0.6665,  ...,  0.3115, -1.7675,  0.6818],
         ...,
         [ 0.0691,  0.2990, -1.4717,  ...,  0.1517,  0.8528,  0.0604],
         [-0.8109,  0.2410, -0.1139,  ...,  1.4509,  0.1836,  0.3064],
         [ 0.5978, -